In [1]:
pip install ultralytics roboflow matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 51.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 58.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.7/805.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 MB 50.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 35.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: idnam╺━━━━━━━━━━━━━━━━━━━━━━  9/21 [kiwisolver]on]headless]
    Found existing

In [4]:
import zipfile, os

ZIP_PATH = "Dent and Scratch.v1i.yolov11.zip"
DATA_DIR = "dent_dataset"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Dataset extracted to:", DATA_DIR)


Dataset extracted to: dent_dataset


In [5]:
yaml_content = """
path: ../dent_dataset
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['dent', 'scratch']
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)


In [ ]:
from ultralytics import YOLO

# Load YOLOv11 base model
model = YOLO("yolo11n.pt")   # nano model (fast). You can use yolo11s.pt for better accuracy.

# Train
results = model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,      # use 'cpu' if no GPU
    workers=2,
    project="dent_yolo",
    name="run1"
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.10 🚀 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA RTX A5000, 24123MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.

In [ ]:
metrics = model.val()
print(metrics)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

log = pd.read_csv("dent_yolo/run1/results.csv")

plt.figure(figsize=(10,4))
plt.plot(log['epoch'], log['metrics/mAP50(B)'], label='mAP50')
plt.plot(log['epoch'], log['metrics/mAP50-95(B)'], label='mAP50-95')
plt.legend()
plt.title("mAP over epochs")
plt.show()


In [ ]:
# Predict on test set
pred = model.predict(
    source="dent_dataset/test/images",
    conf=0.25,
    save=True
)
